<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Assignment 6 (optional, extra credit) — Integer Programming: Pick a Fantasy Football Team

In M5.1 you used binary variables to choose projects: $y = 1$ if we do the project, $y = 0$ if we do not. Here
the projects are football players. In a fantasy football contest you pick a team of real NFL players, and your
team scores points based on how those players do in that week's real games. Every player has a salary, and
the whole team has to fit under a salary cap.

**You do not write any model code.** The model is already here, and a whole scenario is **one line**. There are
three parts. In each one you run a few cells, read what they print, and fill in one answer cell. Each part is
worth one third of the credit.

| Part | The question |
|---|---|
| 1 | Whole players or fractions: why do we need integer programming? |
| 2 | Write two business rules in math, and see what each one costs |
| 3 | Hindsight or forecast: what happens when we pick the team before the games? |

### The model you are solving

A team has seven players: at least 2 running backs (RB), at least 3 wide receivers (WR) and at least 1 tight end
(TE). The seventh player, called the flex, can be any of the three. This is a DraftKings lineup without its
quarterback and its defense, because our file does not have them. A full lineup has a cap of &#36;50,000, so for these
seven spots we use &#36;40,000.

**Decision variables**

$y_i = 1$ if we pick player $i$, and $y_i = 0$ if we do not.

**Objective function**

$$\max \sum_{i} \text{points}_i \, y_i$$

**Constraints**

$$
\begin{aligned}
\sum_{i \in RB} y_i \ge 2, \quad \sum_{i \in WR} y_i \ge 3, \quad \sum_{i \in TE} y_i \ge 1 && \text{(the positions)} \\
\sum_{i} y_i = 7 && \text{(seven players)} \\
\sum_{i} \text{salary}_i \, y_i \le 40{,}000 && \text{(the salary cap)} \\
y_i \in \{0, 1\} \quad \text{for every player } i && \text{(a player is on the team or not)}
\end{aligned}
$$

## Setup (run these three cells)

The first cell installs Pyomo and **cbc**, the solver for models with binary variables. It takes about a minute.

In [ ]:
# install Pyomo and the cbc solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
from pyomo.environ import *

# where the setup cell put the solver
cbc_path = '/content/bin/cbc'

**The players.** The file has one row per player per week for Weeks 2 through 10 of the 2021 NFL season: name,
position, team, opponent, fantasy salary and fantasy points. We are playing the **Week 10** contest. A few players
have a salary of &#36;0 in Week 10, which means they were not in the contest, so the cell drops them.

In [ ]:
all_weeks = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/Assignments/data/nfl_fantasy_2021.csv')

# we are playing the Week 10 contest
players = all_weeks[all_weeks['Week'] == 10]

# drop the players who were not in the contest
players = players[players['Salary'] > 0]
players = players.reset_index(drop=True)

print(len(players), "players in the Week 10 contest")
print(players['Position'].value_counts())

players.head()

**The model.** The code in the next cell is hidden because you do not need to change it (double-click the title
to see it). It is the model at the top of this notebook, plus the two business rules you will switch on in Part 2.

After this cell, a whole scenario is one line:

`pick_team("a name")`

It prints the team, its points and its salary.

In [ ]:
#@title Load the model (just run this cell - the code is hidden on purpose) { display-mode: "form" }
import logging
from IPython.display import display
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

# the team: at least this many at each position, and this many players in all
roster_rules = {"RB": 2, "WR": 3, "TE": 1}
team_size = 7

scoreboard = {}
indices = players.index.values.tolist()
Points = players['Points'].tolist()
Salary = players['Salary'].tolist()
Position = players['Position'].tolist()
Name = players['Player'].tolist()


def pick_team(scenario, salary_cap=40000, whole_players=True, not_together=None, requires=None, forecast=False):
    # empty rule lists when a rule is not switched on
    if not_together is None:
        not_together = []
    if requires is None:
        requires = []

    # check the names first, so a typo gets a helpful message
    for pair in not_together + requires:
        for name in pair:
            if name not in Name:
                print("I cannot find", repr(name), "- check the spelling")
                return

    model = ConcreteModel()

    # decision variables: one per player
    if whole_players:
        model.y = Var(indices, domain=Binary)
    else:
        model.y = Var(indices, domain=UnitInterval)      # fractions of a player are allowed

    model.constraints = ConstraintList()

    # the positions: at least 2 RB, at least 3 WR, at least 1 TE
    for position in roster_rules:
        expr = 0
        for i in indices:
            if Position[i] == position:
                expr += model.y[i]
        model.constraints.add(expr >= roster_rules[position])

    # seven players in all (the seventh is the flex)
    expr = 0
    for i in indices:
        expr += model.y[i]
    model.constraints.add(expr == team_size)

    # the salary cap
    expr = 0
    for i in indices:
        expr += Salary[i] * model.y[i]
    model.constraints.add(expr <= salary_cap)

    # mutually exclusive: y_a + y_b <= 1
    for pair in not_together:
        model.constraints.add(model.y[Name.index(pair[0])] + model.y[Name.index(pair[1])] <= 1)

    # contingency: the first player requires the second, so y_first <= y_second
    for pair in requires:
        model.constraints.add(model.y[Name.index(pair[0])] <= model.y[Name.index(pair[1])])

    # the objective: the points we believe in
    if forecast:
        believed = players['Forecast'].tolist()
    else:
        believed = Points
    expr = 0
    for i in indices:
        expr += believed[i] * model.y[i]
    model.total = Objective(expr=expr, sense=maximize)

    results = SolverFactory('cbc', executable=cbc_path).solve(model)
    status = str(results.solver.termination_condition)
    print("SCENARIO:", scenario, "   (solver:", status, ")")
    if status != "optimal":
        print("no team can meet all of these rules")
        return

    # the team
    picked = []
    share = []
    salary_used = 0
    points_scored = 0
    for i in indices:
        if model.y[i]() > 0.001:
            picked.append(i)
            share.append(round(model.y[i](), 3))
            salary_used += Salary[i] * model.y[i]()
            points_scored += Points[i] * model.y[i]()

    columns = ['Position', 'Player', 'Team', 'Salary', 'Points']
    if forecast:
        columns.append('Forecast')
    roster = players.loc[picked, columns].copy()
    if not whole_players:
        roster['Share of player'] = share
    display(roster.sort_values('Position'))

    if forecast:
        print("the model expected {:.1f} points".format(model.total()))
    print("points actually scored = {:.1f}".format(points_scored))
    print("salary used = ${:,.0f} of ${:,.0f}".format(salary_used, salary_cap))

    scoreboard[scenario] = round(points_scored, 1)


def show_all():
    names = list(scoreboard.keys())
    scores = list(scoreboard.values())
    figure(figsize=(9, 1.2 + 0.5 * len(names)))
    barh(names, scores, color='#E0701A')
    for k in range(len(names)):
        text(scores[k] + 1, k, str(scores[k]), va='center')
    gca().invert_yaxis()
    xlim(0, max(scores) * 1.12)
    xlabel('Fantasy points actually scored in Week 10')
    title('Same players, different choices')
    show()


print("the model is loaded")

# Part 1 · Whole players or fractions

First the base team: the best seven players that fit the positions and the &#36;40,000 cap.

In [ ]:
pick_team("1 base team")

Now remove the integer rule. `whole_players=False` lets $y_i$ be any number from 0 to 1, so the model can buy
part of a player. The last column shows the share of each player it bought.

In [ ]:
pick_team("2 fractions allowed", whole_players=False)

### YOUR ANSWER - Part 1

Double-click this cell and fill it in.

**1a.** The base team scores \_\_\_\_\_\_ points and uses &#36;\_\_\_\_\_\_ of the cap. With fractions allowed, the team
scores \_\_\_\_\_\_ points.

**1b.** Which two players were split, and what share of each did the model buy? You cannot put part of a player on
the field, so why not just round each share to 0 or 1? And why can a whole-number rule never raise the score? Two
or three sentences.

*your sentences*

# Part 2 · Write two rules in math, and see what each one costs

These are two of the rule types from the project selection videos in M5.1, written for players.

| Rule | In words | Type |
|---|---|---|
| A | Deebo Samuel or Stefon Diggs, but not both | Mutually exclusive |
| B | We can only pick Darrel Williams if we also pick his teammate Tyreek Hill | Contingency |

Each cell switches on **one** rule, so you can compare it with the base team from Part 1.

In [ ]:
pick_team("3 rule A: Samuel or Diggs", not_together=[("Deebo Samuel", "Stefon Diggs")])

In [ ]:
pick_team("4 rule B: Williams needs Hill", requires=[("Darrel Williams", "Tyreek Hill")])

### YOUR ANSWER - Part 2

**2a.** Write each rule in math, using $y$ (for example $y_{Samuel}$):

Rule A, Samuel or Diggs, but not both: \_\_\_\_\_\_\_\_\_\_\_\_

Rule B, Williams only if Hill is also on the team: \_\_\_\_\_\_\_\_\_\_\_\_

**2b.** The base team scored \_\_\_\_\_\_ points. With rule A the team scores \_\_\_\_\_\_, so rule A cost \_\_\_\_\_\_ points.
With rule B the team scores \_\_\_\_\_\_, so rule B cost \_\_\_\_\_\_ points. For the rule that cost more, who left the
team and who came in? One or two sentences.

*your sentences*

# Part 3 · Hindsight or forecast

There is a catch in everything so far. We used each player's **actual Week 10 points**, and nobody knows those
when the contest opens. A real team has to be picked before kickoff, using a forecast.

Here is a simple forecast: each player's average points over Weeks 2 through 9. The first cell builds it. The
second cell maximizes the **forecast** in place of the actual points, then shows what that team really scored in
Week 10.

Your final fund is in the same position this week: you picked it from past returns, and it is being scored on days
that nobody had seen.

In [ ]:
# each player's average points in the weeks before our contest
past = all_weeks[all_weeks['Week'] < 10]
past_average = past.groupby('Player')['Points'].mean()

players['Forecast'] = players['Player'].map(past_average)
players['Forecast'] = players['Forecast'].fillna(0).round(2)

# the ten players with the best forecasts, next to what they really scored
players[['Player', 'Position', 'Salary', 'Forecast', 'Points']].sort_values('Forecast', ascending=False).head(10)

In [ ]:
pick_team("5 picked from a forecast", forecast=True)

One picture of every team you built, by the points it really scored.

In [ ]:
show_all()

### YOUR ANSWER - Part 3

**3a.** The model expected \_\_\_\_\_\_ points. The team actually scored \_\_\_\_\_\_ points. The base team in Part 1
scored \_\_\_\_\_\_ points.

**3b.** Why did the forecast team score so much less than the model expected? And what does that tell you about
the base team's score from Part 1? Two or three sentences.

*your sentences*

# Turn it in

1. Click **Runtime → Run all** and wait for it to finish. Scroll down and make sure every part printed a team and
   there are no red error messages.
2. Check that all three **YOUR ANSWER** cells are filled in.
3. **File → Download → Download .ipynb**, and upload that file to Assignment 6 on HuskyCT by the due date shown
   there.

This assignment is optional and counts as extra credit.